In [1]:
import numpy as np
import pandas as pd
from scipy.io import loadmat
from pathlib import Path

import torch
import pickle

from kymatio.torch import Scattering1D

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn import preprocessing
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier


#%matplotlib notebook
import matplotlib.pyplot as plt

In [2]:
base_path = Path('./')

input_path = base_path.joinpath('input_directory')
output_path = base_path.joinpath('output_directory')

In [3]:
X = []
Y = []
age = []
sex = []
for mat_file in input_path.glob('*.mat'):
    print(mat_file, end = "\r")
    mat = loadmat(mat_file)['val'].astype('float32')
    #I dont't normalize the data on purpose; more on that later
    mat /= 1000 # convert to milliVolt scale
    X.append(torch.tensor(mat))

    hea_file = mat_file.parent / (mat_file.stem + '.hea')
    #print(hea_file)
    
    #with hea_file.open() as f: #first line of file is different
        #print(f.readline()) 
    
    df = pd.read_csv(hea_file, sep = ' ', skiprows=1, index_col = 0, header = None) #use pandas to parse header
    Y.append( df.loc['#Dx:',1].split(',') )
    sex.append( df.loc['#Sex:',1] == 'Male' ) #Male == 1, Female == 0
    age.append( float(df.loc['#Age:',1]) )

age = np.clip(np.nan_to_num(np.array(age), nan = 60.),0,120) #replace bad values by guesses
sex = np.array(sex)
G = torch.tensor(np.stack([age,sex], axis = 1).astype('float32')) #additional features; some ages are negative???

t_min = min(x.shape[1] for x in X) #get minimum length in the dataset
t_min

3000

In [4]:
label_set = set() #find the set of different labels
for labels in Y:
    #if len(labels) >= 2: print(labels)
    for label in labels:
        label_set.add(label)

num_to_label = dict((idx,lab) for idx,lab in enumerate(label_set))
label_to_num = dict((lab,idx) for idx,lab in enumerate(label_set))

print(num_to_label)
print(label_to_num)
print(label_set)

{0: 'LBBB', 1: 'PVC', 2: 'STD', 3: 'RBBB', 4: 'Normal', 5: 'PAC', 6: 'STE', 7: 'I-AVB', 8: 'AF'}
{'LBBB': 0, 'PVC': 1, 'STD': 2, 'RBBB': 3, 'Normal': 4, 'PAC': 5, 'STE': 6, 'I-AVB': 7, 'AF': 8}
{'LBBB', 'PVC', 'STD', 'RBBB', 'Normal', 'PAC', 'STE', 'I-AVB', 'AF'}


In [10]:
#generate n out of k encoded array of labels
L = np.zeros((len(Y), len(label_set)), dtype = np.int8)
for idx, labels in enumerate(Y):
    for label in labels:
        L[idx, label_to_num[label]] = 1
L = torch.tensor(L)

In [ ]:
Q = 16
def extract_features(x):
    J = int(np.floor(np.log2(t_min)))-1 #log2 of scattering scale
    S = Scattering1D(J, x.shape[1], Q) #J;T;Q; Q number of filters per octave
    s = S(x).mean(axis = 2) #average over time completly
    return s[:,1:] #do not use DC part of signal; <- this is why i dont't need to remove the mean of the raw data

#x = X[0][:,:t_min]
#print(x.shape)
#print(extract_features(x.contiguous()).shape)

def scatter_features(X):
    Z = []
    for idx, x in enumerate(X):
        z = extract_features(x)
        print(idx, z.shape, end = '   \r')
        Z.append(z.flatten())
    return torch.stack(Z)

if 1:
    Z = scatter_features(X)
    pickle.dump(Z, open('Z'+str(Q)+'.bin','wb'))
else:
    Z = pickle.load(open('Z'+str(Q)+'.bin','rb'))
Z.shape

In [7]:
Z2 = torch.log(Z+1e-12) #after removing DC part all Z are positve -> log gives natural scale
Z2 -= Z2.mean(axis = 1, keepdims = True)#remove mean per batch; this is equivalent to normalising the raw signal; remove the degree of freedom of the gain
Z2 -= Z2.mean(axis = 0, keepdims = True)#remove mean per dimension; this is to get all features to the same scale

G2 = G - G.mean(axis = 0, keepdims = True) #G gets normalised per channel
G2 /= G2.std(axis = 0, keepdims = True)

F = torch.cat([Z2,G2], 1) #merge all features into one 

In [11]:
L_train, L_tmp, F_train, F_tmp, = train_test_split(L,     F,     test_size=0.2, random_state=42)
L_test,  L_val, F_test,  F_val, = train_test_split(L_tmp, F_tmp, test_size=0.5, random_state=42)

print(len(L_train), len(L_test), len(L_val))

5501 688 688


In [17]:
clf = MultiOutputClassifier(LogisticRegression(multi_class = 'ovr', verbose=1, max_iter = 10000, C = 0.1))
clf.fit(F_train.numpy(), L_train.numpy())

print( clf.score(F_train.numpy(), L_train.numpy()) )
print( clf.score(F_test.numpy(),  L_test.numpy() ) )

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    1.3s finished
[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    2.0s finished
[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    2.1s finished
[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    2.4s finished
[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    2.1s finished
[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    2.4s finished
[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_j

0.7118705689874568
0.5290697674418605


In [45]:
np.array(clf.predict_proba(F_test))[:,:,1].shape

(9, 688)

In [25]:
label_set

{'AF', 'I-AVB', 'LBBB', 'Normal', 'PAC', 'PVC', 'RBBB', 'STD', 'STE'}